# Lab 6: Queryable Encryption

SSNs and account numbers stay encrypted client-side, before they ever hit
the wire or disk. You can still query them by equality. Someone with
direct collection access — a DBA, a cloud admin, an attacker who dumps the
data files — sees ciphertext, not plaintext.

This is Atlas and Enterprise Advanced only. MongoDB Community Edition —
and Percona Server for MongoDB, which is built on Community — supports
only a much more limited "Explicit Encryption" mode, where the application
has to manually call encrypt/decrypt on every field, on every read and
write. What this notebook shows — automatic, driver-level encryption
that's completely transparent to application code — has no equivalent
outside Atlas or Enterprise Advanced.

**Setup required (once), local runs only — Colab needs the extra step
below:**
- `pip install "pymongo[encryption]"` (already installed by the bootstrap
  cell)
- Download the **Automatic Encryption Shared Library** (`crypt_shared`) from
  the MongoDB Download Center and set `CRYPT_SHARED_LIB_PATH` in `.env` to
  its path (e.g. `/usr/local/lib/mongo_crypt_v1.dylib` on macOS,
  `mongo_crypt_v1.so` on Linux/Colab).
- The **local master key** generated below is for this demo only. In
  production you'd point `kms_providers` at AWS/Azure/GCP KMS instead, so
  MongoDB itself never has access to the key.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "banking_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)


In [ ]:
# Colab only: crypt_shared isn't preinstalled there. Look up the current
# download URL for this exact distro/arch from MongoDB's own download feed
# (a hardcoded URL+version here goes stale within days as new patch
# releases ship) and download it once per session.
if IN_COLAB and not os.environ.get("CRYPT_SHARED_LIB_PATH"):
    import json, platform, tarfile, urllib.request

    os_release = {}
    with open("/etc/os-release") as f:
        for line in f:
            if "=" in line:
                k, v = line.strip().split("=", 1)
                os_release[k] = v.strip('"')
    target = os_release.get("ID", "ubuntu") + os_release.get("VERSION_ID", "22.04").replace(".", "")
    arch = platform.machine()

    feed = json.loads(urllib.request.urlopen("https://downloads.mongodb.org/current.json").read())
    crypt_url = None
    for v in feed["versions"]:
        match = next(
            (d for d in v.get("downloads", [])
             if d.get("edition") == "enterprise" and d.get("target") == target and d.get("arch") == arch),
            None,
        )
        if match and match.get("crypt_shared"):
            crypt_url = match["crypt_shared"]["url"]
            break
    if crypt_url is None:
        raise RuntimeError(f"No crypt_shared build found for target={target} arch={arch}")

    tarfile_path = "/tmp/crypt_shared.tgz"
    urllib.request.urlretrieve(crypt_url, tarfile_path)
    with tarfile.open(tarfile_path) as tf:
        tf.extractall("/tmp/crypt_shared")
    os.environ["CRYPT_SHARED_LIB_PATH"] = "/tmp/crypt_shared/lib/mongo_crypt_v1.so"
    print("crypt_shared downloaded from", crypt_url)
    print("  ->", os.environ["CRYPT_SHARED_LIB_PATH"])

CRYPT_SHARED_LIB_PATH = os.environ.get("CRYPT_SHARED_LIB_PATH")


In [ ]:
from pymongo import MongoClient
from pymongo.encryption import AutoEncryptionOpts, ClientEncryption
from bson.codec_options import CodecOptions
from bson.binary import STANDARD

# --- DEMO ONLY: local Customer Master Key. Use a real KMS in production. ---
master_key_path = "customer-master-key.txt"
if not os.path.exists(master_key_path):
    with open(master_key_path, "wb") as f:
        f.write(os.urandom(96))
with open(master_key_path, "rb") as f:
    local_master_key = f.read()

kms_providers = {"local": {"key": local_master_key}}
key_vault_namespace = "encryption.__keyVault"


In [ ]:
encrypted_fields = {
    "fields": [
        {"path": "ssn", "bsonType": "string", "queries": [{"queryType": "equality"}]},
        {"path": "account_number", "bsonType": "string"},
    ]
}

auto_encryption_opts = AutoEncryptionOpts(
    kms_providers,
    key_vault_namespace,
    crypt_shared_lib_path=CRYPT_SHARED_LIB_PATH,
)

encrypted_client = MongoClient(ATLAS_URI, auto_encryption_opts=auto_encryption_opts, tlsCAFile=CA_FILE)

client_encryption = ClientEncryption(
    kms_providers=kms_providers,
    key_vault_namespace=key_vault_namespace,
    key_vault_client=encrypted_client,
    codec_options=CodecOptions(uuid_representation=STANDARD),
)

encrypted_coll_name = "customers_encrypted"
encrypted_client[DEMO_DB].drop_collection(encrypted_coll_name)

client_encryption.create_encrypted_collection(
    encrypted_client[DEMO_DB],
    encrypted_coll_name,
    encrypted_fields,
    "local",
    {},
)
print("Encrypted collection ready:", f"{DEMO_DB}.{encrypted_coll_name}")


## Insert and query through the encrypted client

This looks like a completely normal `pymongo` call — the driver encrypts
`ssn` and `account_number` automatically before sending the insert, and
decrypts them automatically on read.


In [ ]:
enc_coll = encrypted_client[DEMO_DB][encrypted_coll_name]

enc_coll.insert_one({
    "name": "Jordan Rivera",
    "ssn": "123-45-6789",
    "account_number": "ACCT-4111-2222-3333",
    "email": "jordan.rivera@example.com",
})

result = enc_coll.find_one({"ssn": "123-45-6789"}, {"__safeContent__": 0})
print("Read through the ENCRYPTED client:")
print(result)


## Now read the same document with a plain client

Same connection string, same collection — but no encryption options. This is
what a DBA running `mongosh`, or anyone with direct database access, actually
sees.


In [ ]:
plain_client = MongoClient(ATLAS_URI, tlsCAFile=CA_FILE)
raw = plain_client[DEMO_DB][encrypted_coll_name].find_one(
    {"name": "Jordan Rivera"}, {"__safeContent__": 0}
)
print("Read through a PLAIN client (no encryption keys):")
print(raw)


`ssn` and `account_number` come back as encrypted BSON binary — unreadable
without the data encryption key, which lives in the key vault and is itself
protected by the (in production, KMS-backed) customer master key.

### Why this matters

- No app-level crypto code — this is standard `pymongo`, just with automatic
  encryption options configured once.
- Still queryable by equality (and range, on server 8.0+) — this isn't
  "encrypt and lose the ability to query," which is the usual tradeoff people
  expect.
- Directly answers the banking insider-threat / compliance question: cloud
  provider staff, a compromised ops account, or a stolen backup file all see
  ciphertext only.
- Swap the `local` KMS provider for AWS/Azure/GCP KMS in production so
  MongoDB never holds the key at all — customer keeps full control.
- Percona Server for MongoDB and Community can encrypt data at rest on
  disk, but that protects against a stolen drive, not a live query. A DBA
  running `mongosh` against a Community/Percona deployment sees every field
  in plaintext. This notebook is the difference.
